In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.home() / "kkbox-churn" / "data" / "parquet"

con = duckdb.connect()
con.execute("SET threads=2")
con.execute("SET memory_limit='10GB'")
con.execute("SET max_temp_directory_size='100GB'")
con.execute("SET preserve_insertion_order=false")

con.execute(f"CREATE OR REPLACE VIEW txn AS SELECT * FROM '{DATA_DIR}/transactions.parquet'")
con.execute(f"CREATE OR REPLACE VIEW txn2 AS SELECT * FROM '{DATA_DIR}/transactions_v2.parquet'")
con.execute(f"CREATE OR REPLACE VIEW train AS SELECT * FROM '{DATA_DIR}/train_v2.parquet'")

print("DuckDB ready")
con.execute("SELECT COUNT(*) FROM txn").fetchone()

DuckDB ready


(21547746,)

In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW txn_all AS
    SELECT * FROM '{DATA_DIR}/transactions.parquet'
    UNION ALL
    SELECT * FROM '{DATA_DIR}/transactions_v2.parquet'
""")

total = con.execute("SELECT COUNT(*) FROM txn_all").fetchone()[0]
print(f"Total transactions (union): {total:,}")

train_users = con.execute(f"""
    SELECT COUNT(DISTINCT msno) 
    FROM txn_all 
    WHERE msno IN (SELECT msno FROM '{DATA_DIR}/train_v2.parquet')
""").fetchone()[0]
print(f"Train users with transaction history: {train_users:,}")
print(f"Missing (no transactions): {970960 - train_users:,}")

Total transactions (union): 22,978,755
Train users with transaction history: 970,960
Missing (no transactions): 0


In [ ]:
con.execute(f"""
    CREATE OR REPLACE VIEW txn_sorted AS
    SELECT *
    FROM txn_all
    WHERE msno IN (SELECT msno FROM '{DATA_DIR}/train_v2.parquet')
    ORDER BY msno, transaction_date
""")

sample = con.execute("""
    SELECT msno, transaction_date, membership_expire_date, 
           is_auto_renew, is_cancel, payment_plan_days, actual_amount_paid
    FROM txn_sorted
    LIMIT 10
""").df()

print(sample.to_string())
print(f"\nColumns: {list(sample.columns)}")

                                           msno  transaction_date  membership_expire_date  is_auto_renew  is_cancel  payment_plan_days  actual_amount_paid
0  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=          20161116                20161215              1          0                 30                  99
1  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=          20161215                20170115              1          0                 30                  99
2  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=          20170115                20170215              1          0                 30                  99
3  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=          20170215                20170315              1          0                 30                  99
4  +++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=          20170315                20170415              1          0                 30                  99
5  +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=          20150131     

In [ ]:
txn_features = con.execute("""
WITH base AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) AS rn,
        ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date ASC)  AS rn_asc,
        STRPTIME(CAST(transaction_date AS VARCHAR), '%Y%m%d')::DATE           AS txn_date_parsed,
        STRPTIME(CAST(membership_expire_date AS VARCHAR), '%Y%m%d')::DATE     AS expire_date_parsed
    FROM txn_sorted
),
collapsed AS (
    SELECT
        msno,
        MAX(CASE WHEN rn = 1 THEN txn_date_parsed END)           AS last_transaction_date,
        MAX(CASE WHEN rn = 1 THEN expire_date_parsed END)         AS last_expire_date,
        MAX(CASE WHEN rn = 1 THEN is_auto_renew END)              AS current_auto_renew,
        MAX(CASE WHEN rn = 1 THEN is_cancel END)                  AS current_is_cancel,
        MAX(CASE WHEN rn = 1 THEN payment_plan_days END)          AS current_plan_days,
        MAX(CASE WHEN rn = 1 THEN actual_amount_paid END)         AS current_amount_paid,
        MAX(CASE WHEN rn = 1 THEN payment_method_id END)          AS current_payment_method,
        MAX(CASE WHEN rn_asc = 1 THEN txn_date_parsed END)        AS first_transaction_date,
        MAX(CASE WHEN rn_asc = 1 THEN actual_amount_paid END)     AS first_amount_paid,
        SUM(CASE WHEN rn <= 3 THEN is_cancel ELSE 0 END)          AS cancel_last3,
        SUM(CASE WHEN rn <= 3 THEN is_auto_renew ELSE 0 END)      AS autorenew_last3,
        SUM(CASE WHEN rn <= 5 THEN is_cancel ELSE 0 END)          AS cancel_last5,
        SUM(CASE WHEN rn <= 5 THEN is_auto_renew ELSE 0 END)      AS autorenew_last5,
        COUNT(*)                                                   AS total_transactions,
        SUM(is_cancel)                                             AS total_cancellations,
        AVG(is_auto_renew)                                         AS auto_renew_rate,
        AVG(is_cancel)                                             AS cancel_rate,
        SUM(actual_amount_paid)                                    AS total_amount_paid,
        AVG(actual_amount_paid)                                    AS avg_amount_paid,
        AVG(CASE WHEN payment_plan_days > 0
                 THEN actual_amount_paid::DOUBLE / payment_plan_days
                 ELSE NULL END)                                    AS pay_per_day,
        AVG(payment_plan_days)                                     AS avg_plan_days,
        MAX(payment_plan_days)                                     AS max_plan_days,
        MAX(CASE WHEN payment_plan_days >= 180 THEN 1 ELSE 0 END)  AS has_long_plan,
        COUNT(DISTINCT payment_method_id)                          AS payment_methods_used,
        SUM(CASE WHEN actual_amount_paid = 0 THEN 1 ELSE 0 END)   AS total_discount,
        MAX(CASE WHEN actual_amount_paid = 0 THEN 1 ELSE 0 END)   AS discount_flag,
        MAX(CASE WHEN is_cancel = 1 AND transaction_date >= 20170301
                 THEN 1 ELSE 0 END)                                AS cancelled_last_30_days,
        MAX(CASE WHEN is_cancel = 1
                 THEN txn_date_parsed ELSE NULL END)               AS last_cancel_date
    FROM base
    GROUP BY msno
),
intervals AS (
    SELECT msno,
        AVG(gap) AS avg_interval_days
    FROM (
        SELECT msno,
            DATE_DIFF('day',
                LAG(txn_date_parsed) OVER (PARTITION BY msno ORDER BY txn_date_parsed),
                txn_date_parsed
            ) AS gap
        FROM base
    )
    WHERE gap IS NOT NULL AND gap > 0
    GROUP BY msno
)
SELECT
    c.msno,
    c.last_transaction_date,
    c.last_expire_date,
    c.current_auto_renew,
    c.current_is_cancel,
    c.current_plan_days,
    c.current_payment_method,
    -- days_since = ref - event (positive = days ago)
    DATE_DIFF('day', c.last_transaction_date, DATE '2017-03-31')   AS days_since_last_transaction,
    DATE_DIFF('day', c.last_cancel_date,      DATE '2017-03-31')   AS days_since_last_cancel,
    DATE_DIFF('day', c.first_transaction_date, DATE '2017-03-31')  AS customer_tenure_days,
    -- days_to_expiry = expire - ref (positive = still active, negative = already expired)
    DATE_DIFF('day', DATE '2017-04-01', c.last_expire_date)        AS days_to_expiry,
    c.total_transactions,
    c.total_cancellations,
    c.auto_renew_rate,
    c.cancel_rate,
    c.total_amount_paid,
    c.avg_amount_paid,
    c.pay_per_day,
    c.avg_plan_days,
    c.max_plan_days,
    c.has_long_plan,
    c.payment_methods_used,
    c.total_discount,
    c.discount_flag,
    c.cancelled_last_30_days,
    c.cancel_last3,
    c.autorenew_last3,
    c.cancel_last5,
    c.autorenew_last5,
    (c.current_amount_paid - c.first_amount_paid)                  AS price_change,
    CASE WHEN c.current_amount_paid > c.first_amount_paid
         THEN 1 ELSE 0 END                                         AS has_price_increase,
    CASE WHEN c.payment_methods_used > 1
         THEN 1 ELSE 0 END                                         AS payment_method_changed,
    CASE
        WHEN c.current_auto_renew = 0 THEN 'off'
        WHEN c.current_auto_renew = 1 AND c.auto_renew_rate < 1.0 THEN 'switched_on'
        ELSE 'always_on'
    END                                                            AS auto_renew_switch,
    CASE WHEN c.current_auto_renew = 0 AND c.auto_renew_rate > 0
         THEN 1 ELSE 0 END                                         AS switched_off_flag,
    i.avg_interval_days
FROM collapsed c
LEFT JOIN intervals i ON c.msno = i.msno
""").df()

print(f"Shape: {txn_features.shape}")
print(f"\nNull counts:\n{txn_features.isnull().sum()[txn_features.isnull().sum() > 0]}")
print(f"\nSample:\n{txn_features[['msno','current_auto_renew','days_since_last_transaction','days_to_expiry','total_transactions','cancel_last3']].head(5).to_string()}")

Shape: (970960, 35)

Null counts:
days_since_last_cancel    745988
pay_per_day                    1
avg_interval_days           8618
dtype: int64

Sample:
                                           msno  current_auto_renew  days_since_last_transaction  days_to_expiry  total_transactions  cancel_last3
0  +/EE6uUPt6X+HpNyL4NiwP24g16v7o9NRXgzaqQ1yF4=                   1                            0              29                   9           0.0
1  +/XcWkTwT2fCFFYwVjPjTbNYzqYbc2UJ8Zs6CbrbCkM=                   1                           14              16                   8           0.0
2  +0lSGTZKwfVjc0s36k7YvJmF1DGqJft289lJCrcG/qI=                   0                           20               9                   7           0.0
3  +25KylkhqkH137FdyMCU0pHUu06ITY/9yOCsyeDl6xw=                   1                            9              21                  14           0.0
4  +3+6UDIIlqrd+JwTUBUX/6eQxl4bvbF3O3rsqt/gszs=                   1                           26              

In [ ]:
patch = con.execute("""
WITH base AS (
    SELECT *
    FROM txn_sorted
)
SELECT
    msno,
    AVG(CASE WHEN is_auto_renew = 1 AND is_cancel = 0 THEN 1.0 ELSE 0.0 END) AS auto_not_cancel_ratio,
    COUNT(DISTINCT payment_plan_days)                                           AS plan_changes
FROM base
GROUP BY msno
""").df()

print(f"Patch shape: {patch.shape}")
print(patch.head(3).to_string())

Patch shape: (970960, 3)
                                           msno  auto_not_cancel_ratio  plan_changes
0  +++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=                    1.0             3
1  ++/ZHqwUNa7U21Qz+zqteiXlZapxey86l6eEorrak/g=                    1.0             2
2  ++0/NopttBsaAn6qHZA2AWWrDg7Me7UOMs1vsyo4tSI=                    1.0             1


In [ ]:
patch2 = con.execute("""
WITH base AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY msno ORDER BY transaction_date DESC) AS rn
    FROM txn_sorted
)
SELECT
    msno,
    AVG(CASE WHEN actual_amount_paid = 0 THEN 1.0 ELSE 0.0 END) AS nopay_ratio,
    MAX(CASE WHEN rn = 1 THEN actual_amount_paid END)             AS t1_amount,
    MAX(CASE WHEN rn = 2 THEN actual_amount_paid END)             AS t2_amount
FROM base
GROUP BY msno
""").df()

print(f"Patch2 shape: {patch2.shape}")
print(patch2.head(3).to_string())
print(f"\nNulls:\n{patch2.isnull().sum()}")

Patch2 shape: (970960, 4)
                                           msno  nopay_ratio  t1_amount  t2_amount
0  ++mdQeVStSHeEN+/Cfg28lCB7I9zRuJEqFc3AM2gArA=          0.0        129        129
1  +1kXppBPHxRLpqYOHuiareditfCUNeWDjjgT3IfFjmk=          0.0        149        149
2  +1p6C1nzhS1+oPhyGOZXCQS1XQxNHB9q4vciNLd9NOk=          0.0         99         99

Nulls:
msno              0
nopay_ratio       0
t1_amount         0
t2_amount      7667
dtype: int64


In [ ]:
txn_features = txn_features.merge(patch2, on='msno', how='left')

print(f"Final shape: {txn_features.shape}")

out_path = DATA_DIR / "txn_features.parquet"
txn_features.to_parquet(out_path, index=False)

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Saved: {out_path}")
print(f"Size: {size_mb:.1f} MB")
print(f"Shape verify: {pd.read_parquet(out_path).shape}")

Final shape: (970960, 38)
Saved: /Users/harshithnr/kkbox-churn/data/parquet/txn_features.parquet
Size: 61.9 MB
Shape verify: (970960, 38)
